# Comparing pipeline exit-code handling in Bash

This notebook compares three mechanisms for capturing or propagating intermediate failures in Bash pipelines: `set -o pipefail`, the `PIPESTATUS` array, and Bash 5.3 shell-context command substitution.

> **Scope** — Bash 5.3.15 (bash53-015, published 2026-06-10) on Linux. macOS `/bin/bash` is 3.2; install 5.3+ via Homebrew for the examples below.

---

**last_verified:** 2026-08-26  
**tool_version:** bash 5.3.15  
**sources:** https://lists.gnu.org/archive/html/bug-bash/2026-06/msg00043.html, https://lists.gnu.org/archive/html/bash-announce/2025-07/msg00000.html

## Purpose

By default, a Bash pipeline reports only the exit status of the last command. In CI and production scripts this is dangerous: `cmd1 | cmd2` returns 0 even if `cmd1` fails, as long as `cmd2` succeeds. This notebook shows three ways to close that gap, when each is appropriate, and the edge cases that still trap practitioners.

## Key terminology

- **Pipeline exit status** — The integer returned by the shell after a pipeline (`cmd1 | cmd2`). Without `pipefail`, this is the exit status of `cmd2` only.
- **`set -o pipefail`** — A shell option that changes the pipeline exit status to the rightmost non-zero status, or zero if all commands succeed.
- **`PIPESTATUS`** — A read-only array populated after a pipeline finishes. `${PIPESTATUS[0]}` is `cmd1`'s status, `${PIPESTATUS[1]}` is `cmd2`'s status, etc.
- **Shell-context substitution** — Bash 5.3 `${ command; }` and `${|command;}` execute in the current shell instead of a subshell. The first captures stdout; the second leaves the last expression's value in `REPLY`.
- **Exit status propagation** — The mechanism by which a failure in one pipeline component influences the pipeline's overall return code.

In [ ]:
#!/usr/bin/env bash

# default-behavior.sh — show that a pipeline hides intermediate failures

false | true

echo "Pipeline exit status: $?"
echo "(Expected: 0, because 'true' is last. 'false' failure is silently swallowed.)"

### Default behavior

Running `false | true` prints `0`. The shell discards `false`'s exit status because `true` is the last command. Any script that relies on `$?` after an unguarded pipeline will miss exactly the failures it most wants to catch.

In [ ]:
#!/usr/bin/env bash

# pipefail-demo.sh — demonstrate set -o pipefail

set -o pipefail

false | true
echo "With pipefail: $?"

true | false
echo "With pipefail (last fails): $?"

true | true
echo "With pipefail (all succeed): $?"

### `set -o pipefail`

With `pipefail` enabled, the pipeline exit status becomes the rightmost non-zero status, or zero if every command succeeds. `false | true` now yields `1`. This is the cheapest fix for most CI scripts because it requires no array handling and works with `set -e`.

**Limitation:** `pipefail` tells you *that* the pipeline failed, not *which* command failed. A three-stage pipeline (`cmd1 | cmd2 | cmd3`) gives you only one bit of information.

In [ ]:
#!/usr/bin/env bash

# pipestatus-demo.sh — demonstrate PIPESTATUS array

set -o pipefail

# Three-stage pipeline with mixed results
false | true | false
echo "Pipeline exit status: $?"
echo "PIPESTATUS per stage: ${PIPESTATUS[*]}"

# Capture per-stage status without pipefail
false | true
echo "Without pipefail: ${PIPESTATUS[*]}"

### `PIPESTATUS`

`PIPESTATUS` is populated after every pipeline. `${PIPESTATUS[*]}` gives the exit status of each stage in order. In the example above, `false | true | false` yields `[1] = 1` even without `pipefail`, because the array records every stage regardless of the pipeline's overall status.

**Important:** `PIPESTATUS` is overwritten by any subsequent pipeline, including command substitution. Capture it immediately: `status=("${PIPESTATUS[@]}")`.

In [ ]:
#!/usr/bin/env bash

# shell-context-demo.sh — Bash 5.3 shell-context substitution

# ${ command; } runs in the current shell and captures stdout.
# Side effects (variables, traps) persist because no subshell is forked.

count=0
result=${
  count=$((count + 1))
  echo "inner=$count"
}
echo "result: $result"
echo "count after substitution: $count"

# ${|command;} leaves the last expression value in REPLY.
# Useful when you want side effects but do not need captured stdout.
count=0
${|
  count=$((count + 1))
  echo "side-effect only"
}
echo "REPLY: $REPLY"
echo "count after REPLY-substitution: $count"

### Shell-context substitution (Bash 5.3)

Bash 5.3 introduced `${ command; }` and `${|command;}`. Both execute in the current shell context instead of a subshell. The first form captures stdout into the variable; the second form evaluates to the last expression's value in `REPLY`.

For pipeline exit-code handling, shell-context substitution matters because traditional `$(cmd1 | cmd2)` forks a subshell where `pipefail` and `PIPESTATUS` behave differently — and variable mutations inside the subshell are lost. `${ cmd1 | cmd2; }` preserves the current shell's options and variables.

**Availability:** Bash 5.3+ only. macOS ships 3.2; Linux distributions moved to 5.3 in 2025–2026.

## Comparison

| Dimension | `set -o pipefail` | `PIPESTATUS` | Shell-context substitution |
|---|---|---|
| Granularity | Pipeline-level (1 bit) | Per-stage (array) | Per-substitution |
| Side-effect preservation | Yes | Yes | Yes (no subshell fork) |
| Minimum Bash version | 3.0 | 3.0 | 5.3 |
| Interaction with `set -e` | Yes — pipeline failure aborts | Yes — array must be read immediately | Yes — failure aborts in current shell |
| Best for | CI scripts that need a boolean fail/pass | Debugging which stage broke | Scripts that need both captured output and mutated state |

## Verify

1. **pipefail:** `bash -c 'set -o pipefail; false | true; echo $?'` — should print `1`.
2. **PIPESTATUS:** `bash -c 'false | true; echo ${PIPESTATUS[*]}'` — should print `1 0`.
3. **Shell-context:** `bash -c 'count=0; result=$((count+=1, count)); echo $count $result'` — requires Bash 5.3; should print `1 1` with `count` mutated in the current shell.

## Common errors

- **`local var=$(cmd)` masks exit status:** `local` itself returns 0, defeating `set -e` on the assignment. Declare and assign on separate lines when the exit code matters.
- **`PIPESTATUS` clobbered by subshells:** Any subsequent pipeline — including `$(...)` or a pipe in `if` — overwrites the array. Copy it immediately: `status=("${PIPESTATUS[@]}")`.
- **`pipefail` inside command substitution:** Pre-4.4 Bash suppresses `errexit` inside `$(...)`; 4.4+ adds `inherit_errexit` to close most gaps, but conditionals and `local`-decl assignments still leak.
- **Assuming `${ cmd; }` works on older Bash:** It is a Bash 5.3 feature only. Guard with `[[ $BASH_VERSION =~ ^5\.[3-9] ]]` or use a subshell with explicit error propagation.

## References

- Bash 5.3 release announcement (2025-07-05): https://lists.gnu.org/archive/html/bash-announce/2025-07/msg00000.html
- Bash 5.3.15 patch note (2026-06-10): https://lists.gnu.org/archive/html/bug-bash/2026-06/msg00043.html
- Production Bash Patterns (exit-status traps, `local` masking, `inherit_errexit`): https://blog.damonkohler.com/Garden/Production-Bash-Patterns